# DM_G1_P0004_integrated_deep

## 0. 학습 범위

- Course code: DM
- Gate: G1
- Phase: P0004
- Topic: G1 통합 심화 — Simplex Tableau, Two-phase, Big-M, Duality Bridge
- Goal: P0001~P0003의 전범위를 통합해 표준형, tableau, pivot, 특수 종료 신호, artificial variable, Big-M, duality, complementary slackness를 연결한다.
- Source basis: `DM_PDF01__04_rag.md`, `DM_PDF05__04_rag.md`, `DM_PDF02__04_rag.md`
- Output language: Korean
- Code mode: hint_only
- Web grounding: no


## 1. 작성 규칙

- 이 노트북에는 최종 정답, 완성 pivot tableau, Phase I 최종 tableau, Big-M 최종 tableau, dual 정답을 쓰지 않는다.
- 각 대형 문제는 현실 언어, 수식/타블로 언어, Solver 언어를 모두 연결해 답한다.
- 모든 답안에는 `node_id`와 `source anchor`를 함께 적는다.
- 코드 셀은 힌트와 시각화 보조용이다. 계산 결과를 그대로 출력해 정답을 드러내지 않는다.
- 답안을 쓴 뒤에만 `DM_G1_P0004_integrated_deep_answer.ipynb`를 열어 비교한다.


## 2. 채점 기준

| 평가 항목 | 배점 |
| --- | ---: |
| 표준형, BFS, tableau 구성 | 15 |
| entering/leaving/ratio/pivot 판정 | 15 |
| 특수 종료 신호: 최적/복수최적/비유계/infeasible | 15 |
| slack/surplus/artificial variable 및 two-phase 해석 | 20 |
| Big-M formulation과 부호 판정 | 15 |
| duality, weak/strong, complementary slackness | 15 |
| Solver mapping, source anchor, node_id | 5 |


## 3. 심화문제 세트


### 심화문제 1. 스마트 물류센터 생산계획 — Simplex Tableau와 특수 종료 신호

스마트 물류센터가 두 종류의 자동분류 모듈 A, B를 조립한다. A 모듈 생산량을 `x1`, B 모듈 생산량을 `x2`라고 한다. A 모듈 1대는 이익 8백만원, B 모듈 1대는 이익 6백만원을 낸다.

```text
Maximize z = 8x1 + 6x2

subject to
2x1 + x2 <= 180      조립라인 시간
x1 + 2x2 <= 160      검사라인 시간
x1 <= 70             A 모듈 최대 수요
x2 <= 60             B 모듈 최대 수요
x1, x2 >= 0
```

요구사항:

1. slack variables `s1, s2, s3, s4`를 도입하여 표준형을 쓰라.
2. 초기 기저변수와 비기저변수를 구분하라.
3. 초기 BFS를 쓰라.
4. 초기 tableau의 objective row를 `z - 8x1 - 6x2 = 0` convention으로 구성하라.
5. Dantzig rule 기준 첫 entering variable을 고르고 이유를 설명하라.
6. ratio test 후보 행을 모두 쓰고, 제외해야 하는 행이 있다면 이유를 쓰라.
7. 첫 leaving variable과 pivot element를 고르라.
8. 첫 pivot 후 새 basis가 무엇인지 쓰라. 단, 완성된 pivot tableau 전체는 작성하지 않는다.
9. 그래프 해법 관점에서 첫 pivot이 어떤 꼭짓점 이동인지 설명하라.
10. 목적함수를 `z = 8x1 + 4x2`로 바꾸면 복수 최적해 가능성이 생기는지 판정하라. 왜 특정 최적 face 위의 점만 최적인지 설명하라.
11. 다음 tableau 신호를 판정하라: “Max 문제에서 `x3`를 entering variable로 선택하면 objective는 개선되지만, `x3` column의 모든 제약행 계수가 0 이하이다.” 이것이 infeasible인지 unbounded인지 설명하라.
12. Solver 관점에서 changing cells, target cell, constraint LHS cells, RHS cells, Simplex LP, nonnegativity option을 설명하라.

node_id:

- `n_DM_PDF01.simplex_tableau`
- `n_DM_PDF01.entering_leaving_variable`
- `n_DM_PDF01.minimum_ratio_test`
- `n_DM_PDF01.pivot_operation`
- `n_DM_PDF01.optimality_test`
- `n_DM_PDF01.multiple_optima`
- `n_DM_PDF01.unbounded_solution`

source anchors:

- `DM_PDF01:p001:L003`
- `DM_PDF01:p002:L003`
- `DM_PDF01:p003:L003`


In [ ]:
# 힌트:
# - <= 제약에는 slack variable을 더한다.
# - >= 제약에는 surplus variable을 빼고 artificial variable을 더한다.
# - entering column의 양수 계수 행만 ratio test 후보로 본다.
# - Phase I objective는 artificial variables의 합을 최소화한다.
# - artificial variable은 최종해에서 0이어야 원문제와 동치다.
# - primal 제약은 dual variable에 대응하고, primal 변수는 dual constraint에 대응한다.


# 시각화 힌트: 스마트 물류센터 LP의 feasible region을 직접 그려 보라.
# 이 셀은 최적해를 표시하지 않는다. 제약선과 실행가능영역만 확인한다.
import numpy as np
import matplotlib.pyplot as plt

x1 = np.linspace(0, 100, 300)
y_line_1 = 180 - 2*x1          # 2x1 + x2 <= 180
y_line_2 = (160 - x1) / 2      # x1 + 2x2 <= 160

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x1, y_line_1)
ax.plot(x1, y_line_2)
ax.axvline(70, color="gray", linestyle="--")
ax.axhline(60, color="gray", linestyle="--")
ax.set_xlim(0, 100)
ax.set_ylim(0, 90)
ax.grid(alpha=0.3)
ax.set_title("문제 1 시각화 힌트: 제약선과 실행가능영역")

# 빈칸으로 남겨둘 것:
# 1) 실행가능영역을 손으로 음영 처리한다.
# 2) 초기 BFS를 표시한다.
# 3) 첫 pivot 이동 방향을 화살표로 표시한다.
plt.show()


### 내 답안 — 심화문제 1

- 표준형:
- 초기 basis / nonbasis:
- 초기 BFS:
- 초기 tableau objective row:
- 첫 entering variable:
- ratio test 후보와 제외 행:
- leaving variable / pivot element:
- 첫 pivot 후 basis:
- 그래프 해법 관점:
- 목적함수 변경 시 복수 최적해 판정:
- unbounded/infeasible 판정:
- Solver mapping:
- node_id:
- source anchors:


### 심화문제 2. 응급 물류 혼합계획 — Two-phase, Surplus, Artificial Variable

재난 대응센터가 두 종류의 응급 키트 `x1`, `x2`를 준비한다. `x1`은 고급 키트, `x2`는 표준 키트다. 고급 키트는 효과 점수 7, 표준 키트는 효과 점수 5를 가진다. 센터는 효과 점수를 최대화하되, 포장공간, 필수 의료성분, 운송박스 균형 조건을 만족해야 한다.

```text
Maximize z = 7x1 + 5x2

subject to
x1 + x2 <= 30          포장공간 제한
3x1 + 2x2 >= 48        필수 의료성분 최소 요구량
x1 + 2x2 = 28          운송박스 균형 조건
x1, x2 >= 0
```

요구사항:

1. 각 제약식에 대해 slack, surplus, artificial variable 중 무엇이 필요한지 판정하라.
2. 표준형을 쓰라.
3. 초기 basis 후보를 쓰라.
4. 왜 두 번째 제약식에는 surplus만으로 초기 BFS가 생기지 않는지 설명하라.
5. 왜 세 번째 등식 제약에는 artificial variable이 필요한지 설명하라.
6. Phase I objective `w`를 쓰라.
7. Phase I에서 `w*=0`이 뜻하는 바를 설명하라.
8. `w*=0`을 원문제 최적목적값 `z*=0`으로 해석하면 왜 틀리는지 설명하라.
9. Phase I 종료 후 artificial variables가 모두 비기저가 되었다고 가정할 때, Phase II로 넘어가는 절차를 설명하라.
10. 다음 학생 답안을 오답 진단하라: “`3x1+2x2>=48`은 `3x1+2x2+s2+a2=48`로 쓰면 된다.”
11. Phase I 최적해에서 artificial variable `a3=2`가 양수로 남는다면 원문제 feasible 여부를 판정하라.
12. Solver 관점에서 이 문제를 직접 푼다면 changing cells, target cell, constraint LHS/RHS cells를 어떻게 둘지 설명하라. 수동 two-phase와 Solver의 차이를 설명하라.

node_id:

- `n_DM_PDF01.surplus_artificial_variable`
- `n_DM_PDF01.two_phase_method`
- `n_DM_PDF05.artificial_variable`

source anchors:

- `DM_PDF01:p004:L002`
- `DM_PDF01:p004:L003`
- `DM_PDF01:p005:L001`
- `DM_PDF01:p006:L001`
- `DM_PDF01:p009:L001`
- `DM_PDF05:p007:L003`


In [ ]:
# 힌트:
# - <= 제약에는 slack variable을 더한다.
# - >= 제약에는 surplus variable을 빼고 artificial variable을 더한다.
# - entering column의 양수 계수 행만 ratio test 후보로 본다.
# - Phase I objective는 artificial variables의 합을 최소화한다.
# - artificial variable은 최종해에서 0이어야 원문제와 동치다.
# - primal 제약은 dual variable에 대응하고, primal 변수는 dual constraint에 대응한다.


# 시각화 힌트: 각 제약식이 어떤 보조변수를 요구하는지 흐름도로 정리하라.
# 이 셀은 최종 표준형을 완성하지 않는다.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.axis("off")
items = [
    ("x1+x2 <= 30", "slack 후보"),
    ("3x1+2x2 >= 48", "surplus + artificial 후보"),
    ("x1+2x2 = 28", "artificial 후보"),
]
for i, (left, right) in enumerate(items):
    y = 0.8 - i*0.28
    ax.text(0.08, y, left, bbox=dict(boxstyle="round", fc="#eef6ff", ec="#335"))
    ax.annotate("", xy=(0.55, y), xytext=(0.36, y), arrowprops=dict(arrowstyle="->"))
    ax.text(0.60, y, right, bbox=dict(boxstyle="round", fc="#fff5dd", ec="#553"))
ax.set_title("문제 2 시각화 힌트: 제약 유형 -> 보조변수")
plt.show()


### 내 답안 — 심화문제 2

- 보조변수 판정표:
- 표준형:
- 초기 basis 후보:
- surplus만으로 초기 BFS가 생기지 않는 이유:
- 등식 제약에 artificial variable이 필요한 이유:
- Phase I objective:
- `w*=0`의 의미:
- `w=0`과 `z=0` 혼동 교정:
- Phase II 전환 절차:
- 학생 답안 오답 진단:
- artificial positive 잔류 판정:
- Solver mapping:
- node_id:
- source anchors:


### 심화문제 3. 친환경 부품 조달계획 — Big-M, Duality, Complementary Slackness

친환경 전자제품 회사가 세 종류의 부품 공급안 `x1, x2, x3`을 조합한다. 각 공급안은 품질지표 Q1, Q2를 일정 수준 이상 만족해야 하며, 총 조달비용을 최소화한다.

```text
Minimize Z = 6x1 + 4x2 + 7x3

subject to
2x1 + x2 + 3x3 >= 30       품질지표 Q1 최소 요구량
x1 + 2x2 + x3 >= 24         품질지표 Q2 최소 요구량
x1, x2, x3 >= 0
```

Part A. Big-M formulation

1. 최소화 문제를 `max -Z` 형태로 바꾸라.
2. 각 `>=` 제약을 surplus/artificial variable 포함 표준형으로 바꾸라.
3. 초기 기저변수를 쓰라.
4. artificial variables `r1, r2`가 왜 필요한지 설명하라.
5. Big-M objective를 일반 optimization convention으로 쓰라. 즉 artificial variable이 커질수록 objective가 나빠지는 방향으로 표현하라.
6. 강의식 tableau convention에서는 Big-M 부호가 다르게 보일 수 있음을 설명하되, 핵심은 artificial variable을 최종해에서 0으로 몰아내는 것임을 명시하라.
7. `r1=0`, `r2=0` 조건이 왜 원문제와의 동치 조건인지 설명하라.
8. `r1` 또는 `r2`가 최종해에서 양수로 남으면 어떤 결론을 내려야 하는지 쓰라.
9. Big-M과 two-phase method의 공통점과 차이점을 표로 비교하라.

Part B. Duality bridge

위 primal minimization problem의 dual을 작성하라.

1. dual variables `y1, y2`를 정의하고 현실 의미를 쓰라.
2. dual objective를 쓰라.
3. `x1, x2, x3` 각각에 대응하는 dual constraints를 쓰라.
4. primal의 RHS와 dual objective coefficient가 어떻게 대응되는지 설명하라.
5. primal 변수 하나가 dual 제약 하나에 대응한다는 점을 명시하라.
6. primal feasible solution `x=(12, 12, 0), Z=120`과 dual feasible solution `y=(1, 1), W=54`가 weak duality를 만족하는지 판정하라.
7. dual feasible region에서 최적 후보 꼭짓점을 찾는 절차를 설명하라. 단, 풀이용 노트북에는 dual 최종해를 쓰지 말 것.
8. strong duality 관점에서 primal 최적값과 dual 최적값의 관계를 설명하라.
9. complementary slackness로 `x1=0`이어야 하는 이유를 설명하라.
10. complementary slackness를 사용해 primal 최적해 후보를 구하라.
11. reduced cost와 shadow price가 각각 어떤 질문에 답하는지 구분하고, 이 문제가 G2 sensitivity analysis로 어떻게 이어지는지 설명하라.
12. Solver 관점에서 이 문제의 changing cells, target cell, constraint LHS/RHS cells, 최소화 설정, nonnegativity option을 설명하라.

node_id:

- `n_DM_PDF05.artificial_variable`
- `n_DM_PDF05.big_m_method`
- `n_DM_PDF05.min_to_max_conversion`
- `n_DM_PDF05.dual_problem`
- `n_DM_PDF05.primal_dual_mapping`
- `n_DM_PDF05.weak_strong_duality`
- `n_DM_PDF05.complementary_slackness`
- `n_DM_PDF02.reduced_cost`
- `n_DM_PDF02.shadow_price`

source anchors:

- `DM_PDF05:p002:L002`
- `DM_PDF05:p003:L009`
- `DM_PDF05:p003:L015`
- `DM_PDF05:p003:L016`
- `DM_PDF05:p004:L008`
- `DM_PDF05:p007:L003`
- `DM_PDF05:p018:L001`
- `DM_PDF05:p023:L002`
- `DM_PDF05:p024:L001`
- `DM_PDF05:p027:L001`
- `DM_PDF05:p032:L002`


In [ ]:
# 힌트:
# - <= 제약에는 slack variable을 더한다.
# - >= 제약에는 surplus variable을 빼고 artificial variable을 더한다.
# - entering column의 양수 계수 행만 ratio test 후보로 본다.
# - Phase I objective는 artificial variables의 합을 최소화한다.
# - artificial variable은 최종해에서 0이어야 원문제와 동치다.
# - primal 제약은 dual variable에 대응하고, primal 변수는 dual constraint에 대응한다.


# 시각화 힌트: dual feasible region을 직접 그려 보라.
# 이 셀은 dual constraints의 구조만 보여 주며 최적점을 표시하지 않는다.
import numpy as np
import matplotlib.pyplot as plt

y1 = np.linspace(0, 4, 300)
c1 = 6 - 2*y1       # 2y1 + y2 <= 6
c2 = (4 - y1) / 2   # y1 + 2y2 <= 4
c3 = 7 - 3*y1       # 3y1 + y2 <= 7

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(y1, c1)
ax.plot(y1, c2)
ax.plot(y1, c3)
ax.set_xlim(0, 4)
ax.set_ylim(0, 4)
ax.grid(alpha=0.3)
ax.set_title("문제 3 시각화 힌트: dual 제약선")

# 빈칸으로 남겨둘 것:
# 1) y1, y2 >= 0을 반영한다.
# 2) 세 부등식을 동시에 만족하는 영역을 찾는다.
# 3) dual objective 방향을 손으로 표시한다.
plt.show()


### 내 답안 — 심화문제 3

- max `-Z` 변환:
- surplus/artificial 포함 표준형:
- 초기 기저변수:
- artificial variable 필요 이유:
- Big-M objective와 부호 판정:
- tableau convention 주의:
- `r1=r2=0` 동치 조건:
- artificial positive 잔류 판정:
- Big-M vs two-phase 비교:
- dual variables와 현실 의미:
- dual objective:
- dual constraints:
- RHS와 dual objective coefficient 대응:
- weak duality 판정:
- strong duality 해석:
- complementary slackness 해석:
- primal 최적해 후보:
- reduced cost / shadow price 연결:
- Solver mapping:
- node_id:
- source anchors:
